# Training-data scaling (Figure 4C)

Top-1 classification accuracy of the `cell_dino` attention model (gene-KO classifier, C = 1,001) as a function of the per-set cell count, for four **training-set sizes** (1.5M → 50M cells). Larger training pools lift the whole accuracy curve; the x-axis is log-scaled so the early-bin gains (10 → 100 cells) stay visible.

Input is a single tidy CSV in the central figure-data directory:

`../../../data/figures/figure_4/figure_4c_training_data_scaling.csv`

| column | meaning |
| --- | --- |
| `training_set` | training-pool label — `1.5M`, `5M`, `15M`, `50M` |
| `training_cells` | training-pool size in cells (numeric, for ordering) |
| `n_cells` | cells per bag (9 bins, 10 → 5,000) |
| `repetition` | evaluation repetition, 0–49 |
| `top1_acc` / `top5_acc` | accuracy for that single repetition |
| `n_classes` | 1,001 (gene KO / NTC) |

One row per (training set, n_cells, repetition) — 4 × 9 × 50 = 1,800 rows.

**Provenance.** Consolidated from four per-run `attn_train{1p5M,5M,15M,50M}.json`
files written by the attention pipeline. Their `mean_accuracy_top*` and
`stderr_top*` fields reproduce exactly from the raw repetitions kept here, so the
merge was lossless and the JSONs were dropped rather than carried alongside. They
remain in git history under `notebooks/figure_4/panel_C_scaling/` up to commit
`b26a6ff` if the raw files are ever needed again. Run config, identical across all
four and recorded here because it lived only in those JSONs:
`n_repetitions = 50`, `n_classes = 1001`, `mixed_channels_mode = True`,
`split_channels = False`, `device = "cuda"`.


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, MultipleLocator, NullFormatter

# Keep text editable in Illustrator (SVG keeps <text> elements; PDF uses TrueType).
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]

FIGURES_DIR = Path("../../../output/figure_4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Data path

A single tidy CSV in the central figure-data directory, one row per
(training set, n_cells, repetition).

In [ ]:
FIGURE_DATA = Path("../../../data/figures/figure_4")
SCALING_CSV = FIGURE_DATA / "figure_4c_training_data_scaling.csv"

## Configuration

Which accuracy column to plot. The colour ramp (`viridis`, dark = largest pool)
is built in the load cell below, once the training sizes are known from the CSV.

In [ ]:
ACC_COL = "top1_acc"   # "top1_acc" or "top5_acc"

## Load

Read the one CSV and average the 50 repetitions within each (training set,
`n_cells`) bin. Runs are ordered by `training_cells` so the colour ramp and the
legend follow pool size rather than string order (`"15M" < "1.5M"` as text).

In [ ]:
scaling = pd.read_csv(SCALING_CSV)

n_classes = int(scaling["n_classes"].iloc[0])
n_reps = scaling.groupby(["training_set", "n_cells"]).size().unique()

# smallest -> largest training pool. Sort on the numeric training_cells, not the
# label: as text "15M" < "1.5M" < "50M" < "5M", which would scramble the ramp.
RUNS = (scaling[["training_set", "training_cells"]]
        .drop_duplicates()
        .sort_values("training_cells")["training_set"]
        .tolist())

# mean over repetitions -> one accuracy per (training set, n_cells)
curves = (scaling.groupby(["training_set", "n_cells"])[ACC_COL]
          .mean()
          .rename("acc")
          .reset_index())
runs = [(label, curves[curves.training_set == label].sort_values("n_cells"))
        for label in RUNS]

# viridis, light -> dark for smallest -> largest training pool
COLORS = cm.viridis(np.linspace(0.9, 0.08, len(RUNS)))

print(f"gene KO: {n_classes} classes | training sizes: {RUNS} | reps/bin: {n_reps}")

## Figure 4C

Single panel: viridis curves by training-set size, log x-axis (labels at 10 / 200 / 5000 with log minor ticks), percentage y-axis (majors every 20%, minors every 10%). Saves an SVG (paper) + PNG.

In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 7.2))

for (label, df), color in zip(runs, COLORS):
    ax.plot(df["n_cells"].to_numpy(dtype=float), df["acc"].to_numpy(dtype=float),
            "-o", color=color, linewidth=6, markersize=19, zorder=3)

# x-axis: log, sparse labels + exponential (log) minor ticks
ax.set_xscale("log")
ax.set_xticks([10, 200, 5000])
ax.set_xticklabels(["10", "200", "5000"])
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10)))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("# cells per bag", fontsize=42)

# y-axis: 0-100%, majors every 20%, minors every 10%
ax.set_ylabel("Classification accuracy", fontsize=42)
ax.set_ylim(0.0, 1.0)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])
ax.yaxis.set_minor_locator(MultipleLocator(0.1))

ax.tick_params(axis="both", which="major", labelsize=36, width=2.5, length=13)
ax.tick_params(axis="both", which="minor", width=2, length=7)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
for sp in ("left", "bottom"):
    ax.spines[sp].set_linewidth(2)

# legend: largest training pool on top; line+marker handles (no error caps).
# The unit lives in the title so every entry reads consistently.
handles = [Line2D([0], [0], color=c, marker="o", markersize=17, linewidth=6, linestyle="-")
           for c in COLORS]
leg = ax.legend(handles[::-1], RUNS[::-1], title="Training set size (cells)",
                fontsize=28, title_fontsize=28, frameon=False,
                loc="upper left", handlelength=1.4)
leg._legend_box.align = "left"

fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_data_scaling.svg", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "training_data_scaling.png", dpi=240, bbox_inches="tight")
plt.show()

## Summary table

Top-1 accuracy per (training set, n_cells) — the values plotted above — with the
standard error over the 50 repetitions. `acc_sem` reproduces the `stderr_top1`
field of the original per-run JSONs.

In [ ]:
rows = []
for label in RUNS:
    sub = scaling[scaling.training_set == label]
    for n, g in sub.groupby("n_cells"):
        vals = g[ACC_COL].to_numpy(dtype=float)
        rows.append({"training_set": label, "n_cells": int(n),
                     "n_reps": int(vals.size),
                     "acc_mean": float(vals.mean()),
                     # matches stderr_top1 in the source JSONs (std ddof=1 / sqrt(n))
                     "acc_sem": float(vals.std(ddof=1) / np.sqrt(vals.size))})

summary = pd.DataFrame(rows)
summary.to_csv(FIGURES_DIR / "training_data_scaling_summary.csv", index=False)
summary